In [ ]:
!pip install openmeteo-requests requests-cache retry-requests pandas plotly -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 915.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.6/775.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.0/396.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 21.6 MB/s eta 0:00:00


In [ ]:
#stater code provided by Open-Meteo
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 33.8703,
	"longitude": -84.2748,
	"daily": ["sunrise", "sunset", "daylight_duration", "uv_index_max"],
	"hourly": ["temperature_2m", "apparent_temperature", "precipitation_probability", "rain", "showers", "snow_depth", "wind_speed_10m", "relative_humidity_2m", "snowfall"],
	"current": ["temperature_2m", "is_day", "precipitation", "rain", "showers", "snowfall", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m"],
	"temperature_unit": "fahrenheit",
	"wind_speed_unit": "mph",
	"precipitation_unit": "inch",
	"timezone": "America/New_York"
}
responses = openmeteo.weather_api(url, params = params)
response = responses[0]

print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")

# --- Current conditions ---
current = response.Current()
current_conditions = {
    "time": pd.to_datetime(current.Time(), unit='s', utc=True).tz_convert('America/New_York'),
    "temperature_f": current.Variables(0).Value(),
    "is_day": current.Variables(1).Value(),
    "precipitation_in": current.Variables(2).Value(),
    "rain_in": current.Variables(3).Value(),
    "showers_in": current.Variables(4).Value(),
    "snowfall_in": current.Variables(5).Value(),
    "wind_speed_mph": current.Variables(6).Value(),
    "wind_direction_deg": current.Variables(7).Value(),
    "wind_gusts_mph": current.Variables(8).Value(),
}
print("\nCurrent conditions:", current_conditions)

# --- Hourly forecast ---
hourly = response.Hourly()
hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	),
    "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
    "apparent_temperature": hourly.Variables(1).ValuesAsNumpy(),
    "precipitation_probability": hourly.Variables(2).ValuesAsNumpy(),
    "rain": hourly.Variables(3).ValuesAsNumpy(),
    "showers": hourly.Variables(4).ValuesAsNumpy(),
    "snow_depth": hourly.Variables(5).ValuesAsNumpy(),
    "wind_speed_10m": hourly.Variables(6).ValuesAsNumpy(),
    "relative_humidity_2m": hourly.Variables(7).ValuesAsNumpy(),
    "snowfall": hourly.Variables(8).ValuesAsNumpy(),
}
df_forecast_hourly = pd.DataFrame(data=hourly_data)
df_forecast_hourly["date"] = df_forecast_hourly["date"].dt.tz_convert("America/New_York")
print("\nHourly forecast\n", df_forecast_hourly.head())

# --- Daily forecast (sunrise/sunset/UV) ---
daily = response.Daily()
daily_data = {
	"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	),
    "sunrise": pd.to_datetime(daily.Variables(0).ValuesInt64AsNumpy(), unit="s", utc=True).tz_convert("America/New_York"),
    "sunset": pd.to_datetime(daily.Variables(1).ValuesInt64AsNumpy(), unit="s", utc=True).tz_convert("America/New_York"),
    "daylight_duration": daily.Variables(2).ValuesAsNumpy(),
    "uv_index_max": daily.Variables(3).ValuesAsNumpy(),
}
df_forecast_daily = pd.DataFrame(data=daily_data)
df_forecast_daily["date"] = df_forecast_daily["date"].dt.tz_convert("America/New_York")
print("\nDaily forecast\n", df_forecast_daily)

Coordinates: 33.88145446777344°N -84.27667236328125°E
Elevation: 286.0 m asl

Current conditions: {'time': Timestamp('2026-07-07 14:30:00-0400', tz='America/New_York'), 'temperature_f': 89.59190368652344, 'is_day': 1.0, 'precipitation_in': 0.0, 'rain_in': 0.0, 'showers_in': 0.0, 'snowfall_in': 0.0, 'wind_speed_mph': 7.687602519989014, 'wind_direction_deg': 261.6341857910156, 'wind_gusts_mph': 11.856100082397461}

Hourly forecast
                        date  temperature_2m  apparent_temperature  \
0 2026-07-07 00:00:00-04:00       74.921898             82.352150   
1 2026-07-07 01:00:00-04:00       74.201897             82.141663   
2 2026-07-07 02:00:00-04:00       73.931900             80.818314   
3 2026-07-07 03:00:00-04:00       72.401901             78.877495   
4 2026-07-07 04:00:00-04:00       72.041901             77.823540   

   precipitation_probability  rain  showers  snow_depth  wind_speed_10m  \
0                        6.0   0.0      0.0         0.0        2.767016   
1

In [ ]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import plotly.graph_objects as go
from datetime import date, timedelta # Import date and timedelta from datetime module

# --- Re-establish the API client ---
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# --- Re-fetch historical data ---
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 33.8703,
    "longitude": -84.2748,
    "start_date": (date.today() - timedelta(days=2 * 365)).isoformat(), # set start_date to two years before today
    "end_date": date.today().isoformat(), # end_date= today
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "temperature_2m_mean",
        "relative_humidity_2m_mean",
        "precipitation_sum"
    ],
    "temperature_unit": "fahrenheit",
    "precipitation_unit": "inch",
    "timezone": "America/New_York"
}

responses = openmeteo.weather_api(url, params=params)
response = responses[0]

daily = response.Daily()
daily_data = {
    "date": pd.date_range(
        start=pd.to_datetime(daily.Time(), unit="s", utc=True),
        end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=daily.Interval()),
        inclusive="left"
    ),
    "temp_max_f": daily.Variables(0).ValuesAsNumpy(),
    "temp_min_f": daily.Variables(1).ValuesAsNumpy(),
    "temp_mean_f": daily.Variables(2).ValuesAsNumpy(),
    "humidity_mean_pct": daily.Variables(3).ValuesAsNumpy(),
    "precip_sum_in": daily.Variables(4).ValuesAsNumpy(),
}

In [ ]:
df = pd.DataFrame(data=daily_data)
df["date"] = pd.to_datetime(df["date"]).dt.tz_convert("America/New_York")
df = df.sort_values("date").reset_index(drop=True)

df.shape

(731, 6)

In [ ]:
df.head()

,date,temp_max_f,temp_min_f,temp_mean_f,humidity_mean_pct,precip_sum_in
0,2024-07-07 00:00:00-04:00,88.520004,74.029999,78.931244,86.185844,1.157480
1,2024-07-08 00:00:00-04:00,89.419998,73.220001,80.183754,81.787529,0.043307
2,2024-07-09 00:00:00-04:00,88.160004,73.040001,79.955009,80.107445,0.149606
3,2024-07-10 00:00:00-04:00,88.339996,72.230003,78.863747,67.008354,0.011811
4,2024-07-11 00:00:00-04:00,87.529999,68.360001,77.941254,58.498707,0.000000


In [ ]:
df.isnull().sum()

,0
date,0
temp_max_f,0
temp_min_f,0
temp_mean_f,0
humidity_mean_pct,0
precip_sum_in,0


In [ ]:
numeric_cols = ["temp_max_f", "temp_min_f", "temp_mean_f", "humidity_mean_pct", "precip_sum_in"]
df[numeric_cols] = df[numeric_cols].interpolate(method="linear")

In [ ]:
#check null after interpolation(predicting values between two data points)
#this value SHOULD be zero for all columns
df[numeric_cols].isnull().sum()

,0
temp_max_f,0
temp_min_f,0
temp_mean_f,0
humidity_mean_pct,0
precip_sum_in,0


In [ ]:
#derived columns using datetime module
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year
df["day_of_year"] = df["date"].dt.dayofyear
df["temp_mean_7d_avg"] = df["temp_mean_f"].rolling(window=7, center=True).mean()
df["humidity_7d_avg"] = df["humidity_mean_pct"].rolling(window=7, center=True).mean() # Added this line to create the humidity_7d_avg column

df.head()

,date,temp_max_f,temp_min_f,temp_mean_f,humidity_mean_pct,precip_sum_in,month,year,day_of_year,temp_mean_7d_avg,humidity_7d_avg
0,2024-07-07 00:00:00-04:00,88.520004,74.029999,78.931244,86.185844,1.157480,7,2024,189,NaN,NaN
1,2024-07-08 00:00:00-04:00,89.419998,73.220001,80.183754,81.787529,0.043307,7,2024,190,NaN,NaN
2,2024-07-09 00:00:00-04:00,88.160004,73.040001,79.955009,80.107445,0.149606,7,2024,191,NaN,NaN
3,2024-07-10 00:00:00-04:00,88.339996,72.230003,78.863747,67.008354,0.011811,7,2024,192,79.863394,69.55685
4,2024-07-11 00:00:00-04:00,87.529999,68.360001,77.941254,58.498707,0.000000,7,2024,193,80.647680,65.05374


In [ ]:
import plotly.graph_objects as go

#theme for graphs
CHART_FONT = dict(
    family="Inter, Segoe UI, Arial, sans-serif",
    size=13,
    color="#3b3b3b"
)

TITLE_FONT = dict(
    family="Inter, Segoe UI, Arial, sans-serif",
    size=24,
    color="#111111"
)

AXIS_COLOR = "#6B7280"
GRID_COLOR = "#ECEFF3"
BACKGROUND = "#FFFFFF"

COLORS = [
    "#2563EB",   # Blue
    "#10B981",   # Emerald
    "#F59E0B",   # Amber
    "#EF4444",   # Red
    "#8B5CF6",   # Purple
    "#06B6D4",   # Cyan
]

def style_layout(fig, title, yaxis_title="", height=550):
    fig.update_layout(
        title=dict(
            text=f"<b>{title}</b>",
            x=0.02,
            xanchor="left",
            font=TITLE_FONT
        ),
        font=CHART_FONT,
        template="simple_white",
        plot_bgcolor=BACKGROUND,
        paper_bgcolor=BACKGROUND,
        height=height,
        margin=dict(l=70, r=40, t=80, b=60),
        hovermode="x unified",
        hoverlabel=dict(
            bgcolor="white",
            bordercolor="#D9D9D9",
            font_size=13,
            font_family="Inter"
        ),
        legend=dict(
            orientation="h",
            y=1.08,
            x=1,
            xanchor="right",
            bgcolor="rgba(255,255,255,0)",
            font=dict(size=12)
        ),
        colorway=COLORS,
        xaxis=dict(
            showgrid=False,
            showline=True,
            linewidth=1,
            linecolor="#D6D6D6",
            ticks="outside",
            tickcolor="#D6D6D6",
            ticklen=6,
            tickfont=dict(size=12, color=AXIS_COLOR),
            title_font=dict(size=14)
        ),
        yaxis=dict(
            title=yaxis_title,
            showgrid=True,
            gridcolor=GRID_COLOR,
            gridwidth=1,
            showline=True,
            linewidth=1,
            linecolor="#D6D6D6",
            zeroline=False,
            tickfont=dict(size=12, color=AXIS_COLOR),
            title_font=dict(size=14)
        )
    )
    return fig

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["date"], y=df["temp_max_f"],
    name="Daily High", line=dict(color="#fc6060", width=2),
    opacity=0.3, hovertemplate="High: %{y:.1f}°F<extra></extra>"
))

fig.add_trace(go.Scatter(
    x=df["date"], y=df["temp_min_f"],
    name="Daily Low", line=dict(color="#4b82fa", width=2),
    opacity=0.3, hovertemplate="Low: %{y:.1f}°F<extra></extra>"
))

fig.add_trace(go.Scatter(
    x=df["date"], y=df["temp_mean_7d_avg"],
    name="7-Day Avg", line=dict(color="#73c98d", width=3),
    hovertemplate="7-Day Avg: %{y:.1f}°F<extra></extra>"
))

fig = style_layout(fig, "Alpharetta, GA — Temperature Trends (Last 2 Years)", "Temperature (°F)")
fig.show()

In [ ]:
fig_humidity = go.Figure()

fig_humidity.add_trace(go.Scatter(
    x=df["date"], y=df["humidity_mean_pct"],
    name="Daily Mean", line=dict(color="#06B6D4", width=1),
    opacity=0.3, hovertemplate="Humidity: %{y:.1f}%<extra></extra>"
))

fig_humidity.add_trace(go.Scatter(
    x=df["date"], y=df["humidity_7d_avg"],
    name="7-Day Avg", line=dict(color="#0E7490", width=3),
    hovertemplate="7-Day Avg: %{y:.1f}%<extra></extra>"
))

fig_humidity = style_layout(fig_humidity, "Alpharetta, GA — Humidity Trends (Last 2 Years)", "Relative Humidity (%)")
fig_humidity.show()

In [ ]:
fig_precip = go.Figure()

fig_precip.add_trace(go.Bar(
    x=df["date"], y=df["precip_sum_in"],
    name="Daily Precipitation",
    marker_color="#5f9ccf",
    marker_line_width=1, # Increased line width for bar borders
    marker_line_color="#3a6c92", # Added a color for the bar borders
    opacity=0.9,
    hovertemplate="Precip: %{y:.2f} in<extra></extra>"
))

fig_precip = style_layout(fig_precip, "Alpharetta, GA — Daily Precipitation (Last 2 Years)", "Precipitation (in)")
fig_precip.update_layout(showlegend=False)
fig_precip.show()

In [ ]:
monthly_precip = df.groupby([df["date"].dt.year.rename('year'), df["date"].dt.month.rename('month')])["precip_sum_in"].sum().reset_index()
monthly_precip["year_month"] = monthly_precip["year"].astype(str) + "-" + monthly_precip["month"].astype(str).str.zfill(2)

monthly_precip["avg_line"] = monthly_precip["precip_sum_in"].mean()

fig_monthly_precip = go.Figure()

fig_monthly_precip.add_trace(go.Bar(
    x=monthly_precip["year_month"], y=monthly_precip["precip_sum_in"],
    name="Monthly Total",
    marker_color="#57a8b3",
    marker_line_width=0,
    opacity=0.9,
    hovertemplate="%{x}<br>Total: %{y:.2f} in<extra></extra>"
))

fig_monthly_precip.add_trace(go.Scatter(
    x=monthly_precip["year_month"], y=monthly_precip["avg_line"],
    name="2-Year Monthly Avg", mode="lines",
    line=dict(color="#c90202", width=3, dash="dash"),
    hovertemplate="Avg: %{y:.2f} in<extra></extra>"
))

fig_monthly_precip = style_layout(fig_monthly_precip, "Alpharetta, GA — Monthly Precipitation Totals", "Total Precipitation (in)")
fig_monthly_precip.update_layout(xaxis_tickangle=-45)
fig_monthly_precip.show()

In [ ]:
import calendar

df["month_name"] = df["month"].apply(lambda x: calendar.month_abbr[x])
month_order = list(calendar.month_abbr)[1:]

fig_box = go.Figure()

fig_box.add_trace(go.Box(
    x=df["month_name"], y=df["temp_mean_f"],
    name="Mean Temp",
    marker_color="#2563EB",
    boxmean=True
))

fig_box.update_xaxes(categoryorder="array", categoryarray=month_order)
fig_box = style_layout(fig_box, "Alpharetta, GA — Monthly Temperature Distribution", "Temperature (°F)")
fig_box.update_layout(showlegend=False)
fig_box.show()

In [ ]:
import numpy as np

corr_cols = ["temp_max_f", "temp_min_f", "temp_mean_f", "humidity_mean_pct", "precip_sum_in"]
corr_matrix = df[corr_cols].corr()

fig_corr = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale=[[0, "#EF4444"], [0.5, "#FFFFFF"], [1, "#2563EB"]],
    zmid=0,
    text=np.round(corr_matrix.values, 2),
    texttemplate="%{text}",
    textfont=dict(size=12),
    hovertemplate="%{x} vs %{y}: %{z:.2f}<extra></extra>"
))

fig_corr = style_layout(fig_corr, "Alpharetta, GA — Variable Correlation Matrix", height=550)
fig_corr.update_layout(showlegend=False)
fig_corr.show()

In [ ]:
# Map every date onto a shared reference year (2024, a leap year, so Feb 29 works for all years)
df["plot_date"] = pd.to_datetime("2024-" + df["date"].dt.strftime("%m-%d"))

fig_yoy = go.Figure()

for i, yr in enumerate(sorted(df["year"].unique())):
    yr_data = df[df["year"] == yr].sort_values("date")
    fig_yoy.add_trace(go.Scatter(
        x=yr_data["plot_date"], y=yr_data["temp_mean_7d_avg"],
        name=str(yr),
        line=dict(color=COLORS[i % len(COLORS)], width=2.5),
        hovertemplate=f"{yr}: " + "%{y:.1f}°F<extra></extra>"
    ))

fig_yoy = style_layout(fig_yoy, "Alpharetta, GA — Year-over-Year Temperature Comparison", "7-Day Avg Temp (°F)")
fig_yoy.update_xaxes(
    tickformat="%b",       # show just month abbreviations (Jan, Feb, ...)
    dtick="M1",            # one tick per month
    hoverformat="%b %d"    # nice hover label
)
fig_yoy.show()

In [ ]:
fig_hist = go.Figure()

fig_hist.add_trace(go.Histogram(
    x=df["temp_mean_f"], name="Mean Temp",
    marker_color="#76ab63", opacity=0.95, nbinsx=40,
    hovertemplate="Range: %{x}<br>Count: %{y}<extra></extra>"
))

fig_hist.add_vline(
    x=df["temp_mean_f"].mean(), line_dash="dash", line_color="#a30000", line_width=2,
    annotation_text=f"Mean: {df['temp_mean_f'].mean():.1f}°F", annotation_position="top"
)

fig_hist = style_layout(fig_hist, "Alpharetta, GA — Temperature Distribution", "Frequency")
fig_hist.update_xaxes(title_text="Mean Temperature (°F)")
fig_hist.update_layout(showlegend=False)
fig_hist.show()

In [ ]:
df["week"] = df["date"].dt.isocalendar().week
df["weekday"] = df["date"].dt.weekday  # 0=Mon

fig_cal = go.Figure()

for yr in sorted(df["year"].unique()):
    yr_df = df[df["year"] == yr]
    fig_cal.add_trace(go.Heatmap(
        x=yr_df["week"], y=yr_df["weekday"],
        z=yr_df["temp_mean_f"],
        colorscale="RdYlBu_r",
        showscale=bool(yr == sorted(df["year"].unique())[-1]), # Explicitly cast to Python bool
        name=str(yr),
        hovertemplate="Temp: %{z:.1f}°F<extra></extra>",
        colorbar=dict(title="°F")
    ))

fig_cal.update_yaxes(
    tickmode="array", tickvals=list(range(7)),
    ticktext=["Mon","Tue","Wed","Thu","Fri","Sat","Sun"], autorange="reversed"
)
fig_cal = style_layout(fig_cal, "Alpharetta, GA — Daily Temperature Calendar Heatmap", height=350)
fig_cal.update_xaxes(title_text="Week of Year")
fig_cal.show()

In [ ]:
# Temperature range (daily volatility indicator)
df["temp_range_f"] = df["temp_max_f"] - df["temp_min_f"]

# Cyclical encoding of day-of-year (captures seasonality without a hard Jan 1 boundary)
df["day_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
df["day_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365.25)

# Lag features — yesterday's weather, strong predictors for tomorrow
df["temp_mean_lag1"] = df["temp_mean_f"].shift(1)
df["temp_mean_lag2"] = df["temp_mean_f"].shift(2)
df["humidity_lag1"] = df["humidity_mean_pct"].shift(1)
df["precip_lag1"] = df["precip_sum_in"].shift(1)

# Rolling volatility (std dev) — captures unstable/transitional weather periods
df["temp_std_7d"] = df["temp_mean_f"].rolling(window=7).std()

# Heating/Cooling Degree Days (base 65°F, standard energy/climate metric)
df["hdd"] = (65 - df["temp_mean_f"]).clip(lower=0)
df["cdd"] = (df["temp_mean_f"] - 65).clip(lower=0)

# Target variable for Phase 2: tomorrow's mean temperature
df["temp_mean_next_day"] = df["temp_mean_f"].shift(-1)

print("New columns added:")
print(df[["date","temp_range_f","day_sin","day_cos","temp_mean_lag1","temp_std_7d","hdd","cdd","temp_mean_next_day"]].tail(10))

New columns added:
                         date  temp_range_f   day_sin   day_cos  \
721 2026-06-28 00:00:00-04:00     18.360001  0.062318 -0.998056   
722 2026-06-29 00:00:00-04:00     20.070000  0.045141 -0.998981   
723 2026-06-30 00:00:00-04:00     17.460007  0.027950 -0.999609   
724 2026-07-01 00:00:00-04:00     15.029999  0.010751 -0.999942   
725 2026-07-02 00:00:00-04:00     20.070000 -0.006451 -0.999979   
726 2026-07-03 00:00:00-04:00     14.129997 -0.023651 -0.999720   
727 2026-07-04 00:00:00-04:00     18.449997 -0.040844 -0.999166   
728 2026-07-05 00:00:00-04:00     18.540001 -0.058026 -0.998315   
729 2026-07-06 00:00:00-04:00     18.360001 -0.075190 -0.997169   
730 2026-07-07 00:00:00-04:00     23.760002 -0.092331 -0.995728   

     temp_mean_lag1  temp_std_7d  hdd        cdd  temp_mean_next_day  
721       80.982498     3.669415  0.0  17.992493           85.144997  
722       82.992493     4.695746  0.0  20.144997           84.331253  
723       85.144997     4.7179

In [ ]:
fig_scatter = go.Figure()

fig_scatter.add_trace(go.Scatter(
    x=df["temp_mean_f"], y=df["humidity_mean_pct"],
    mode="markers",
    marker=dict(
        size=6, color=df["month"], colorscale="RdYlBu_r",
        showscale=True, colorbar=dict(title="Month"),
        line=dict(width=0.5, color="white")
    ),
    hovertemplate="Temp: %{x:.1f}°F<br>Humidity: %{y:.1f}%<extra></extra>"
))

fig_scatter = style_layout(fig_scatter, "Alpharetta, GA — Temperature vs. Humidity", "Relative Humidity (%)")
fig_scatter.update_xaxes(title_text="Mean Temperature (°F)")
fig_scatter.update_layout(showlegend=False)
fig_scatter.show()

In [ ]:
df_sorted = df.sort_values("date")
df["hdd_cumulative"] = df.groupby("year")["hdd"].cumsum()
df["cdd_cumulative"] = df.groupby("year")["cdd"].cumsum()

fig_degree_days = go.Figure()

fig_degree_days.add_trace(go.Scatter(
    x=df["date"], y=df["hdd_cumulative"], name="Cumulative Heating Degree Days",
    line=dict(color="#EF4444", width=2.5),
    hovertemplate="HDD: %{y:.0f}<extra></extra>"
))

fig_degree_days.add_trace(go.Scatter(
    x=df["date"], y=df["cdd_cumulative"], name="Cumulative Cooling Degree Days",
    line=dict(color="#2563EB", width=2.5),
    hovertemplate="CDD: %{y:.0f}<extra></extra>"
))

fig_degree_days = style_layout(fig_degree_days, "Alpharetta, GA — Cumulative Heating & Cooling Degree Days", "Degree Days (Base 65°F)")
fig_degree_days.show()

In [ ]:
extreme_hot = (df["temp_max_f"] >= 90).sum()
extreme_cold = (df["temp_min_f"] <= 32).sum()
rainy_days = (df["precip_sum_in"] > 0.1).sum()
heavy_rain_days = (df["precip_sum_in"] > 1.0).sum()

extreme_data = pd.DataFrame({
    "category": ["90°F+ Days", "Freezing Days (≤32°F)", "Rainy Days (>0.1in)", "Heavy Rain Days (>1in)"],
    "count": [extreme_hot, extreme_cold, rainy_days, heavy_rain_days]
})

fig_extreme = go.Figure()

fig_extreme.add_trace(go.Bar(
    x=extreme_data["category"], y=extreme_data["count"],
    marker_color=["#EF4444", "#2563EB", "#71d2e3", "#9771f0"],
    marker_line_width=0, opacity=0.9,
    text=extreme_data["count"], textposition="outside",
    hovertemplate="%{x}: %{y} days<extra></extra>"
))

fig_extreme = style_layout(fig_extreme, "Alpharetta, GA — Extreme Weather Day Counts (2 Years)", "Number of Days")
fig_extreme.update_layout(showlegend=False)
fig_extreme.show()

In [ ]:
feature_cols = [
    "temp_mean_f", "temp_max_f", "temp_min_f", "temp_range_f",
    "humidity_mean_pct", "precip_sum_in", "temp_mean_lag1", "temp_mean_lag2",
    "humidity_lag1", "precip_lag1", "temp_std_7d", "day_sin", "day_cos",
    "hdd", "cdd"
]

target_corr = df[feature_cols + ["temp_mean_next_day"]].corr()["temp_mean_next_day"].drop("temp_mean_next_day")
target_corr = target_corr.sort_values()

fig_target_corr = go.Figure()

fig_target_corr.add_trace(go.Bar(
    x=target_corr.values, y=target_corr.index,
    orientation="h",
    marker_color=["#EF4444" if v < 0 else "#2563EB" for v in target_corr.values],
    marker_line_width=0, opacity=0.9,
    hovertemplate="%{y}: %{x:.2f}<extra></extra>"
))

fig_target_corr = style_layout(fig_target_corr, "Feature Correlation with Next-Day Mean Temperature", "Correlation Coefficient", height=500)
fig_target_corr.update_layout(showlegend=False)
fig_target_corr.show()

In [ ]:
#historical wind data (separate from earlier archive call)
url = "https://archive-api.open-meteo.com/v1/archive"
wind_params = {
    "latitude": 33.8703,
    "longitude": -84.2748,
    "start_date": "2023-07-06",
    "end_date": "2025-07-06",
    "daily": ["wind_speed_10m_max", "wind_direction_10m_dominant"],
    "wind_speed_unit": "mph",
    "timezone": "America/New_York"
}

wind_responses = openmeteo.weather_api(url, params=wind_params)
wind_response = wind_responses[0]
wind_daily = wind_response.Daily()

wind_data = {
    "date": pd.date_range(
        start=pd.to_datetime(wind_daily.Time(), unit="s", utc=True),
        end=pd.to_datetime(wind_daily.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=wind_daily.Interval()),
        inclusive="left"
    ),
    "wind_speed_max_mph": wind_daily.Variables(0).ValuesAsNumpy(),
    "wind_direction_dominant": wind_daily.Variables(1).ValuesAsNumpy(),
}

df_wind = pd.DataFrame(data=wind_data)
df_wind["date"] = pd.to_datetime(df_wind["date"]).dt.tz_convert("America/New_York")

print(df_wind.shape)
df_wind.head()

(732, 3)


,date,wind_speed_max_mph,wind_direction_dominant
0,2023-07-06 00:00:00-04:00,9.608688,292.517578
1,2023-07-07 00:00:00-04:00,8.325127,298.187988
2,2023-07-08 00:00:00-04:00,9.368731,282.411255
3,2023-07-09 00:00:00-04:00,12.219836,275.759705
4,2023-07-10 00:00:00-04:00,7.618947,325.988556


In [ ]:
# bin directions into 16 compass sectors and speed into bands
compass_labels = ["N","NNE","NE","ENE","E","ESE","SE","SSE",
                   "S","SSW","SW","WSW","W","WNW","NW","NNW"]

df_wind["direction_bin"] = pd.cut(
    df_wind["wind_direction_dominant"] % 360,
    bins=np.arange(-11.25, 360+11.25, 22.5),
    labels=compass_labels,  # Removed the "N_wrap" label
    ordered=False
)
# The replace line is no longer needed as "N_wrap" is not created.
# df_wind["direction_bin"] = df_wind["direction_bin"].replace("N_wrap", "N")

speed_bins = [0, 5, 10, 15, 20, 100]
speed_labels = ["0-5 mph", "5-10 mph", "10-15 mph", "15-20 mph", "20+ mph"]
df_wind["speed_bin"] = pd.cut(df_wind["wind_speed_max_mph"], bins=speed_bins, labels=speed_labels)

wind_rose_data = df_wind.groupby(["direction_bin", "speed_bin"], observed=True).size().reset_index(name="count")

In [ ]:
# wind rose
fig_windrose = go.Figure()

speed_colors = ["#DBEAFE", "#93C5FD", "#3B82F6", "#305cd9", "#062270"]

for speed_label, color in zip(speed_labels, speed_colors):
    subset = wind_rose_data[wind_rose_data["speed_bin"] == speed_label]

    counts_by_direction = subset.set_index("direction_bin")["count"].reindex(compass_labels, fill_value=0)

    fig_windrose.add_trace(go.Barpolar(
        r=counts_by_direction.values,# radius or number of occurences
        theta=counts_by_direction.index, #theta= directions
        name=speed_label, marker_color=color,
        hovertemplate="%{theta}: %{r} days<extra></extra>"
    ))

fig_windrose.update_layout(
    title=dict(text="<b>Alpharetta, GA — Wind Rose (Speed & Direction, 2 Years)</b>", font=TITLE_FONT, x=0.02),
    font=CHART_FONT,
    polar=dict(
        radialaxis=dict(showgrid=True, gridcolor=GRID_COLOR),
        angularaxis=dict(direction="clockwise", rotation=90)
    ),
    legend=dict(orientation="h", y=-0.1, x=0.5, xanchor="center"),
    height=600,
    paper_bgcolor=BACKGROUND,
    barmode="stack"
)

fig_windrose.show()

In [ ]:
from statsmodels.tsa.stattools import acf, pacf

# cleaned up mean temp series
temp_series = df["temp_mean_f"].dropna()

n_lags = 14
acf_vals = acf(temp_series, nlags=n_lags)
pacf_vals = pacf(temp_series, nlags=n_lags)

# 95% confidence interval bound
conf_bound = 1.96 / np.sqrt(len(temp_series))

In [ ]:
from plotly.subplots import make_subplots

fig_acf = make_subplots(rows=1, cols=2, subplot_titles=("Autocorrelation (ACF)", "Partial Autocorrelation (PACF)"))

fig_acf.add_trace(go.Bar(
    x=list(range(n_lags+1)), y=acf_vals,
    marker_color="#3b75f5", marker_line_width=0,
    hovertemplate="Lag %{x}: %{y:.3f}<extra></extra>", showlegend=False
), row=1, col=1)

fig_acf.add_trace(go.Bar(
    x=list(range(n_lags+1)), y=pacf_vals,
    marker_color="#10B981", marker_line_width=0,
    hovertemplate="Lag %{x}: %{y:.3f}<extra></extra>", showlegend=False
), row=1, col=2)

# confidence bounds
for col in [1, 2]:
    fig_acf.add_hline(y=conf_bound, line_dash="dash", line_color="#EF4444", line_width=1.5, row=1, col=col)
    fig_acf.add_hline(y=-conf_bound, line_dash="dash", line_color="#EF4444", line_width=1.5, row=1, col=col)

fig_acf.update_layout(
    title=dict(text="<b>Temperature Autocorrelation — How Many Lag Days Matter</b>", font=TITLE_FONT, x=0.02),
    font=CHART_FONT,
    height=450,
    plot_bgcolor=BACKGROUND,
    paper_bgcolor=BACKGROUND,
    margin=dict(l=60, r=40, t=100, b=60)
)

fig_acf.update_xaxes(title_text="Lag (days)", showgrid=False, showline=True, linecolor="#D6D6D6")
fig_acf.update_yaxes(title_text="Correlation", showgrid=True, gridcolor=GRID_COLOR, showline=True, linecolor="#D6D6D6")

fig_acf.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# features to analyze
model_features = [
    "temp_mean_lag1", "temp_mean_lag2",
    "humidity_lag1", "precip_lag1",
    "temp_std_7d", "day_sin", "day_cos",
    "hdd", "cdd"
]

target = "temp_mean_next_day"

# remove rows with NaN
model_df = df[model_features + [target, "date"]].dropna().reset_index(drop=True)

print(f"Modeling dataset shape: {model_df.shape}")
print(f"Date range: {model_df['date'].min()} to {model_df['date'].max()}")
model_df.head()

Modeling dataset shape: (724, 11)
Date range: 2024-07-13 00:00:00-04:00 to 2026-07-06 00:00:00-04:00


,temp_mean_lag1,temp_mean_lag2,humidity_lag1,precip_lag1,temp_std_7d,day_sin,day_cos,hdd,cdd,temp_mean_next_day,date
0,80.738747,77.941254,55.972546,0.000000,1.473519,-0.211276,-0.977426,0.0,17.430000,84.421249,2024-07-13 00:00:00-04:00
1,82.430000,80.738747,57.337524,0.015748,2.184299,-0.228058,-0.973648,0.0,19.421249,83.779999,2024-07-14 00:00:00-04:00
2,84.421249,82.430000,54.664074,0.000000,2.462234,-0.244772,-0.969581,0.0,18.779999,80.360001,2024-07-15 00:00:00-04:00
3,83.779999,84.421249,60.254395,0.165354,2.433756,-0.261414,-0.965227,0.0,15.360001,81.500000,2024-07-16 00:00:00-04:00
4,80.360001,83.779999,73.237579,0.082677,2.201379,-0.277979,-0.960587,0.0,16.500000,77.116257,2024-07-17 00:00:00-04:00


In [ ]:
# sort by date
model_df = model_df.sort_values("date").reset_index(drop=True)

# 0.2 as test set(most recent)
split_idx = int(len(model_df) * 0.8)

X = model_df[model_features]
y = model_df[target]

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test = model_df["date"].iloc[split_idx:]

print(f"Training set: {X_train.shape[0]} days ({model_df['date'].iloc[0].date()} to {model_df['date'].iloc[split_idx-1].date()})")
print(f"Test set: {X_test.shape[0]} days ({model_df['date'].iloc[split_idx].date()} to {model_df['date'].iloc[-1].date()})")

Training set: 579 days (2024-07-13 to 2026-02-10)
Test set: 145 days (2026-02-11 to 2026-07-06)


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

# Inspect coefficients — which features the model is leaning on
coef_df = pd.DataFrame({
    "feature": model_features,
    "coefficient": model.coef_
}).sort_values("coefficient", key=abs, ascending=False)

print("Model coefficients (sorted by influence):")
print(coef_df)
print(f"\nIntercept: {model.intercept_:.3f}")

Model coefficients (sorted by influence):
          feature  coefficient
6         day_cos    -4.125488
7             hdd    -0.947257
8             cdd     0.916580
5         day_sin    -0.853557
3     precip_lag1    -0.310910
0  temp_mean_lag1    -0.309292
1  temp_mean_lag2     0.145799
4     temp_std_7d    -0.054891
2   humidity_lag1    -0.053528

Intercept: 79.263


In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(" test set model performance")
print(f"  MAE:  {mae:.2f}°F   (avg error per prediction)")
print(f"  RMSE: {rmse:.2f}°F  (penalizes larger misses)")
print(f"  R²:   {r2:.3f}     (variance explained, 1.0 = perfect)")

 test set model performance
  MAE:  3.29°F   (avg error per prediction)
  RMSE: 4.32°F  (penalizes larger misses)
  R²:   0.828     (variance explained, 1.0 = perfect)


In [ ]:
fig_pred = go.Figure()

fig_pred.add_trace(go.Scatter(
    x=dates_test, y=y_test, name="Actual Temp",
    line=dict(color="#111111", width=2.5),
    hovertemplate="Actual: %{y:.1f}°F<extra></extra>"
))

fig_pred.add_trace(go.Scatter(
    x=dates_test, y=y_pred, name="Predicted Temp",
    line=dict(color="#2563EB", width=2.5, dash="dot"),
    hovertemplate="Predicted: %{y:.1f}°F<extra></extra>"
))

fig_pred = style_layout(fig_pred, "Alpharetta, GA — Predicted vs. Actual Next-Day Temperature (Test Set)", "Temperature (°F)")
fig_pred.show()

In [ ]:
residuals = y_test.values - y_pred

fig_resid = go.Figure()

fig_resid.add_trace(go.Scatter(
    x=dates_test, y=residuals, mode="markers",
    marker=dict(size=6, color="#8B5CF6", line=dict(width=0.5, color="white")),
    hovertemplate="Residual: %{y:.2f}°F<extra></extra>"
))

fig_resid.add_hline(y=0, line_dash="dash", line_color="#EF4444", line_width=2)

fig_resid = style_layout(fig_resid, "Prediction Residuals Over Time (Actual − Predicted)", "Residual (°F)")
fig_resid.update_layout(showlegend=False)
fig_resid.show()

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

# some models benefit from scaled features while tree-based models don't need it
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Linear Regression": (LinearRegression(), X_train, X_test),
    "Ridge Regression": (Ridge(alpha=1.0), X_train_scaled, X_test_scaled),
    "Lasso Regression": (Lasso(alpha=0.1), X_train_scaled, X_test_scaled),
    "Random Forest": (RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42), X_train, X_test),
    "Gradient Boosting": (GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42), X_train, X_test),
    "SVR (RBF)": (SVR(kernel="rbf", C=10, epsilon=0.5), X_train_scaled, X_test_scaled),
}

results = []
predictions = {}

for name, (mdl, X_tr, X_te) in models.items():
    mdl.fit(X_tr, y_train)
    preds = mdl.predict(X_te)
    predictions[name] = preds

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    results.append({"model": name, "MAE": mae, "RMSE": rmse, "R2": r2})

results_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
print(results_df)

               model       MAE      RMSE        R2
0  Linear Regression  3.288201  4.322316  0.828293
1   Ridge Regression  3.303458  4.348506  0.826206
2   Lasso Regression  3.467610  4.637586  0.802331
3          SVR (RBF)  3.502523  4.705039  0.796539
4  Gradient Boosting  3.659807  4.806403  0.787678
5      Random Forest  3.802090  5.065804  0.764141


In [ ]:
fig_compare = go.Figure()

fig_compare.add_trace(go.Bar(
    x=results_df["model"], y=results_df["MAE"],
    name="MAE (°F)", marker_color="#2563EB",
    marker_line_width=0, opacity=0.9,
    hovertemplate="%{x}<br>MAE: %{y:.2f}°F<extra></extra>"
))

fig_compare.add_trace(go.Bar(
    x=results_df["model"], y=results_df["RMSE"],
    name="RMSE (°F)", marker_color="#EF4444",
    marker_line_width=0, opacity=0.9,
    hovertemplate="%{x}<br>RMSE: %{y:.2f}°F<extra></extra>"
))

fig_compare = style_layout(fig_compare, "Model Comparison — Prediction Error (Lower is Better)", "Error (°F)")
fig_compare.update_layout(barmode="group")
fig_compare.show()

In [ ]:
fig_r2 = go.Figure()

fig_r2.add_trace(go.Bar(
    x=results_df["model"], y=results_df["R2"],
    marker_color=COLORS[:len(results_df)],
    marker_line_width=0, opacity=0.9,
    text=results_df["R2"].round(3), textposition="outside",
    hovertemplate="%{x}<br>R²: %{y:.3f}<extra></extra>"
))

fig_r2 = style_layout(fig_r2, "Model Comparison — R² Score (Higher is Better)", "R² Score")
fig_r2.update_layout(showlegend=False)
fig_r2.show()

In [ ]:
top3 = results_df.head(3)["model"].tolist()

fig_top3 = go.Figure()

fig_top3.add_trace(go.Scatter(
    x=dates_test, y=y_test, name="Actual",
    line=dict(color="#111111", width=3),
    hovertemplate="Actual: %{y:.1f}°F<extra></extra>"
))

line_styles = ["dot", "dash", "dashdot"]
for i, name in enumerate(top3):
    fig_top3.add_trace(go.Scatter(
        x=dates_test, y=predictions[name], name=name,
        line=dict(color=COLORS[i+1], width=2, dash=line_styles[i]),
        hovertemplate=f"{name}: " + "%{y:.1f}°F<extra></extra>"
    ))

fig_top3 = style_layout(fig_top3, "Top 3 Models — Predicted vs. Actual Next-Day Temperature", "Temperature (°F)")
fig_top3.show()

In [ ]:
#best performing model
final_model = model  # linear regression
final_predictions = y_pred
print(f"MAE:  {mean_absolute_error(y_test, final_predictions):.2f}°F")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, final_predictions)):.2f}°F")
print(f"R²:   {r2_score(y_test, final_predictions):.3f}")

MAE:  3.29°F
RMSE: 4.32°F
R²:   0.828


In [ ]:
today = pd.Timestamp.now(tz="America/New_York").normalize()

# Open-Meteo's hourly data blends recent actuals + short-term forecast for today
today_hourly = df_forecast_hourly[df_forecast_hourly["date"].dt.date == today.date()]

today_mean_temp = today_hourly["temperature_2m"].mean()
today_mean_humidity = today_hourly["relative_humidity_2m"].mean()
today_total_precip = today_hourly["rain"].sum() + today_hourly["showers"].sum()

print(f"todays mean temp: {today_mean_temp:.1f}°F")
print(f"todays mean humidity: {today_mean_humidity:.1f}%")
print(f"todays total precip: {today_total_precip:.2f} in")

todays mean temp: 80.5°F
todays mean humidity: 68.5%
todays total precip: 0.00 in


In [ ]:
yesterday_date = (today - pd.Timedelta(days=1)).date()
yesterday_row = df[df["date"].dt.date == yesterday_date]

yesterday_mean_temp = (
    yesterday_row["temp_mean_f"].values[0]
    if not yesterday_row.empty
    else df["temp_mean_f"].iloc[-1]
)

print(f"{yesterday_mean_temp:.1f}")

79.5


In [ ]:
last6_days = df[df["date"].dt.date >= (today.date() - pd.Timedelta(days=6))]["temp_mean_f"].tolist()
last7_with_today = last6_days + [today_mean_temp]
temp_std_7d_live = np.std(last7_with_today, ddof=1)

temp_std_7d_live

np.float64(1.8868975399776364)

In [ ]:
day_of_year_today = today.dayofyear
day_sin_today = np.sin(2 * np.pi * day_of_year_today / 365.25)
day_cos_today = np.cos(2 * np.pi * day_of_year_today / 365.25)

hdd_today = max(0, 65 - today_mean_temp)
cdd_today = max(0, today_mean_temp - 65)

In [ ]:
live_features = pd.DataFrame([{
    "temp_mean_lag1": today_mean_temp,
    "temp_mean_lag2": yesterday_mean_temp,
    "humidity_lag1": today_mean_humidity,
    "precip_lag1": today_total_precip,
    "temp_std_7d": temp_std_7d_live,
    "day_sin": day_sin_today,
    "day_cos": day_cos_today,
    "hdd": hdd_today,
    "cdd": cdd_today
}])[model_features]

tomorrow_prediction = final_model.predict(live_features)[0]
tomorrow_date = (today + pd.Timedelta(days=1)).date()


print(f"  Predicted Mean for {tomorrow_date}")
print(f"  {tomorrow_prediction:.1f}°F")

  Predicted Mean for 2026-07-08
  80.6°F


In [ ]:
tomorrow_hourly = df_forecast_hourly[df_forecast_hourly["date"].dt.date == tomorrow_date]
openmeteo_forecast_mean = tomorrow_hourly["temperature_2m"].mean()

print(f"Model prediction:{tomorrow_prediction:.1f}°F")
print(f"Open-Meteo's forecast:{openmeteo_forecast_mean:.1f}°F")
print(f"Difference:{abs(tomorrow_prediction - openmeteo_forecast_mean):.1f}°F")

Model prediction:80.6°F
Open-Meteo's forecast:81.1°F
Difference:0.5°F
